# 00 – Check your setup

Run every cell from top to bottom: **Run All** (▶▶ at the top), or `Shift+Enter` on each cell.

If VS Code asks you to *Select Kernel*, choose **Python Environments → Python 3.12 (`/usr/local/bin/python`)**.

## 1. Versions

In [ ]:
import sys, platform
import qiskit, qiskit_aer, numpy, matplotlib

print("Python     ", sys.version.split()[0], platform.machine())
print("Qiskit     ", qiskit.__version__)
print("Qiskit Aer ", qiskit_aer.__version__)
print("NumPy      ", numpy.__version__)
print("Matplotlib ", matplotlib.__version__)

## 2. Build a Bell circuit

$H$ on qubit 0, then CNOT (control 0, target 1), then measure both:

$$|00\rangle \xrightarrow{H \text{ on } q_0} \tfrac{1}{\sqrt2}(|00\rangle+|01\rangle) \xrightarrow{\text{CNOT}} \tfrac{1}{\sqrt2}(|00\rangle+|11\rangle)$$

(Kets here use **Qiskit order** $|q_1 q_0\rangle$: qubit 0 is the rightmost digit.)

**Predict before you run:** what fraction of shots should be `00`, `01`, `10`, `11`?

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
qc.draw("mpl")

## 3. Run it on the simulator

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
job = sim.run(transpile(qc, sim), shots=1024, seed_simulator=42)
counts = job.result().get_counts()
print(counts)

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts, title="Bell state, 1024 shots")

## 4. Same thing with a primitive (exact statevector) and the state itself

In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

res = StatevectorSampler(seed=42).run([qc], shots=1024).result()
print("Sampler counts:", res[0].data.c.get_counts())

bell = qc.remove_final_measurements(inplace=False)
Statevector(bell).draw("latex")

## ⚠️ Old Qiskit code you will find online (and from AI)

Qiskit 1.0 and 2.0 removed several things. AI tools and older tutorials still write them. **If you see any of these, the code is outdated and will fail:**

| Outdated (will error) | Use this instead (Qiskit 2.x) |
|---|---|
| `from qiskit import execute` / `execute(qc, backend)` | `tqc = transpile(qc, sim)` then `sim.run(tqc, shots=1024).result()` |
| `from qiskit import Aer` / `Aer.get_backend('qasm_simulator')` | `from qiskit_aer import AerSimulator` then `sim = AerSimulator()` |
| `from qiskit import IBMQ` / `IBMQ.load_account()` | `from qiskit_ibm_runtime import QiskitRuntimeService` (only for real hardware) |
| `qiskit.providers.ibmq` | `qiskit_ibm_runtime` |
| `BasicAer`, `qiskit.test`, `qiskit.opflow` | removed – use `qiskit_aer`, `qiskit.quantum_info` |
| `Sampler()` / `Estimator()` from `qiskit.primitives` | `StatevectorSampler()` / `StatevectorEstimator()` (or `qiskit_aer.primitives.SamplerV2`) |

Two current patterns that always work here:

```python
# Pattern A: simulator backend
from qiskit import transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()

# Pattern B: primitives (exact statevector sampling, no Aer needed)
from qiskit.primitives import StatevectorSampler
result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.c.get_counts()   # 'c' = name of your classical register
```

**Bit order:** Qiskit prints bitstrings with qubit 0 on the **right**. `'01'` means q0 = 1, q1 = 0.

## 5. Final check

In [ ]:
ok = set(counts) <= {"00", "11"} and abs(counts.get("00", 0) - 512) < 120
if ok:
    print("✅ If you can see the circuit drawing and the histogram above, you're ready for the problem sets!")
else:
    print("❌ Something is off – take a screenshot of this notebook and send it to Mr. Bayntun.")